# Projet 3 - Dispatch d'une centrale

Notebook autonome pour le problème de base et les extensions.

Contenu :
- chargement robuste des données locales du repo ;
- validation sur le cas 24 heures ;
- résolution du problème de base sur 168 heures ;
- tableaux des décisions optimales et fonctions de valeur ;
- planning on/off et synthèse des résultats ;
- extensions 1, 2 et 3 ;
- export automatique vers `dispatch_results.xlsx`.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
DATA_XLS = ROOT / "Project3data.xls"
VALIDATION_XLSM = ROOT / "Project3code_Student.xlsm"
OUTPUT_XLSX = ROOT / "dispatch_results.xlsx"

assert DATA_XLS.exists(), f"Missing file: {DATA_XLS}"
assert VALIDATION_XLSM.exists(), f"Missing file: {VALIDATION_XLSM}"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
DAYS = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]
NEG_INF = -1e15

print(f"Working directory: {ROOT}")
print(f"Data file: {DATA_XLS.name}")
print(f"Validation file: {VALIDATION_XLSM.name}")


Working directory: D:\realoption
Data file: Project3data.xls
Validation file: Project3code_Student.xlsm


## Fonctions utilitaires et solveurs DP

Les fonctions ci-dessous reprennent la logique correcte du solveur, avec des chemins locaux et des sorties structurées pour le notebook.


In [2]:
def load_data(filepath=DATA_XLS):
    df = pd.read_excel(filepath, sheet_name="WeeklySchedule", header=None)
    margins = np.zeros(168)
    demands = np.zeros(168)
    startups = np.zeros(168)

    for day_idx in range(7):
        for hour_idx in range(24):
            row = hour_idx + 2
            t = day_idx * 24 + hour_idx
            margins[t] = float(df.iloc[row, 1 + day_idx])
            demands[t] = float(df.iloc[row, 10 + day_idx])
            startups[t] = float(df.iloc[row, 19 + day_idx])

    df_stoch = pd.read_excel(filepath, sheet_name="StochasticDemand", header=None)
    stoch_demand = []
    for hour_idx in range(24):
        row = hour_idx + 2
        D1 = float(df_stoch.iloc[row, 1])
        p1 = float(df_stoch.iloc[row, 2])
        D2 = float(df_stoch.iloc[row, 3])
        p2 = float(df_stoch.iloc[row, 4])
        stoch_demand.append((D1, p1, D2, p2))

    return margins, demands, startups, stoch_demand


def load_24h_validation(filepath=VALIDATION_XLSM):
    df = pd.read_excel(filepath, sheet_name="24hours", header=None)
    margins_24 = np.zeros(24)
    demands_24 = np.full(24, 100.0)
    startups_24 = np.full(24, 400.0)
    ref_V_s0 = np.zeros(25)
    ref_V_s1 = np.zeros(25)
    ref_x_s0 = np.zeros(25)
    ref_x_s1 = np.zeros(25)

    for i in range(24):
        margins_24[i] = float(df.iloc[i + 1, 1])

    for i in range(25):
        ref_x_s0[i] = float(df.iloc[i + 1, 6]) if pd.notna(df.iloc[i + 1, 6]) else 0
        ref_x_s1[i] = float(df.iloc[i + 1, 7]) if pd.notna(df.iloc[i + 1, 7]) else 0
        ref_V_s0[i] = float(df.iloc[i + 1, 8]) if pd.notna(df.iloc[i + 1, 8]) else 0
        ref_V_s1[i] = float(df.iloc[i + 1, 9]) if pd.notna(df.iloc[i + 1, 9]) else 0

    return margins_24, demands_24, startups_24, ref_V_s0, ref_V_s1, ref_x_s0, ref_x_s1


def solve_base_dp(margins, demands, startups):
    T = len(margins)
    V = np.zeros((T + 1, 2))
    X = np.zeros((T, 2), dtype=int)
    V[T, 0] = 0.0
    V[T, 1] = NEG_INF

    for t in range(T - 1, -1, -1):
        m = margins[t]
        D = demands[t]
        f = startups[t]

        val_stay_off = V[t + 1, 0]
        val_turn_on = m * D - f + V[t + 1, 1]
        if val_turn_on > val_stay_off:
            V[t, 0] = val_turn_on
            X[t, 0] = 1
        else:
            V[t, 0] = val_stay_off
            X[t, 0] = 0

        val_stay_on = m * D + V[t + 1, 1]
        val_turn_off = V[t + 1, 0]
        if val_stay_on >= val_turn_off:
            V[t, 1] = val_stay_on
            X[t, 1] = 0
        else:
            V[t, 1] = val_turn_off
            X[t, 1] = -1

    return V, X


def extract_schedule(X, s0=0):
    T = X.shape[0]
    states = np.zeros(T + 1, dtype=int)
    decisions = np.zeros(T, dtype=int)
    states[0] = s0

    for t in range(T):
        s = states[t]
        x = X[t, s]
        decisions[t] = x

        if s == 0 and x == 1:
            states[t + 1] = 1
        elif s == 1 and x == -1:
            states[t + 1] = 0
        else:
            states[t + 1] = s

    return states, decisions


def solve_ext1_min_runtime(margins, demands, startups, min_hours=10):
    T = len(margins)
    n_states = min_hours + 1
    V = np.full((T + 1, n_states), NEG_INF)
    X = np.zeros((T, n_states), dtype=int)
    V[T, 0] = 0.0

    for t in range(T - 1, -1, -1):
        m = margins[t]
        D = demands[t]
        f = startups[t]

        val_off = V[t + 1, 0]
        val_on = m * D - f + V[t + 1, 1]
        if val_on > val_off:
            V[t, 0] = val_on
            X[t, 0] = 1
        else:
            V[t, 0] = val_off
            X[t, 0] = 0

        for k in range(1, min_hours):
            V[t, k] = m * D + V[t + 1, k + 1]
            X[t, k] = 0

        val_stay = m * D + V[t + 1, min_hours]
        val_turn_off = V[t + 1, 0]
        if val_stay >= val_turn_off:
            V[t, min_hours] = val_stay
            X[t, min_hours] = 0
        else:
            V[t, min_hours] = val_turn_off
            X[t, min_hours] = -1

    return V, X


def extract_schedule_ext1(X, min_hours=10):
    T = X.shape[0]
    states_detail = np.zeros(T + 1, dtype=int)
    states_binary = np.zeros(T + 1, dtype=int)
    decisions = np.zeros(T, dtype=int)

    for t in range(T):
        s = states_detail[t]
        x = X[t, s]
        decisions[t] = x

        if s == 0 and x == 1:
            states_detail[t + 1] = 1
        elif s == 0 and x == 0:
            states_detail[t + 1] = 0
        elif 1 <= s < min_hours:
            states_detail[t + 1] = s + 1
        elif s == min_hours and x == -1:
            states_detail[t + 1] = 0
        else:
            states_detail[t + 1] = min_hours

        states_binary[t + 1] = 1 if states_detail[t + 1] > 0 else 0

    return states_binary, decisions, states_detail


def solve_ext2_production_cap(margins, demands, startups, max_mwh=16500, step=50):
    T = len(margins)
    n_cum = max_mwh // step + 1
    V = np.full((T + 1, 2, n_cum), NEG_INF)
    X = np.zeros((T, 2, n_cum), dtype=int)
    V[T, 0, :] = 0.0
    V[T, 1, :] = NEG_INF

    for t in range(T - 1, -1, -1):
        m = margins[t]
        D = demands[t]
        f = startups[t]
        d_step = int(D / step)

        for c in range(n_cum):
            val_off = V[t + 1, 0, c]
            new_c = c + d_step
            val_on = m * D - f + V[t + 1, 1, new_c] if new_c < n_cum else NEG_INF
            if val_on > val_off:
                V[t, 0, c] = val_on
                X[t, 0, c] = 1
            else:
                V[t, 0, c] = val_off
                X[t, 0, c] = 0

            val_stay = m * D + V[t + 1, 1, new_c] if new_c < n_cum else NEG_INF
            val_turn_off = V[t + 1, 0, c]
            if val_stay >= val_turn_off:
                V[t, 1, c] = val_stay
                X[t, 1, c] = 0
            else:
                V[t, 1, c] = val_turn_off
                X[t, 1, c] = -1

    return V, X


def extract_schedule_ext2(X, demands, step=50):
    T = X.shape[0]
    states = np.zeros(T + 1, dtype=int)
    cum = np.zeros(T + 1, dtype=int)
    decisions = np.zeros(T, dtype=int)

    for t in range(T):
        s = states[t]
        c = cum[t]
        x = X[t, s, c]
        decisions[t] = x
        d_step = int(demands[t] / step)

        if s == 0 and x == 1:
            states[t + 1] = 1
            cum[t + 1] = c + d_step
        elif s == 1 and x == 0:
            states[t + 1] = 1
            cum[t + 1] = c + d_step
        elif s == 1 and x == -1:
            states[t + 1] = 0
            cum[t + 1] = c
        else:
            states[t + 1] = 0
            cum[t + 1] = c

    return states, decisions, cum


def solve_ext3_stochastic(margins, demands, startups, stoch_demand, max_mwh=16500, step=50):
    T = len(margins)
    n_cum = max_mwh // step + 1
    V = np.full((T + 1, 2, n_cum), NEG_INF)
    X = np.zeros((T, 2, n_cum), dtype=int)
    V[T, 0, :] = 0.0
    V[T, 1, :] = NEG_INF

    for t in range(T - 1, -1, -1):
        m = margins[t]
        f = startups[t]
        day = t // 24
        hour_in_day = t % 24
        if day == 6:
            D1, p1, D2, p2 = stoch_demand[hour_in_day]
            scenarios = [(D1, p1), (D2, p2)]
        else:
            scenarios = [(demands[t], 1.0)]

        for c in range(n_cum):
            ev_off = V[t + 1, 0, c]

            ev_on = 0.0
            feasible_on = True
            for D_val, prob in scenarios:
                new_c = c + int(D_val / step)
                if new_c < n_cum:
                    ev_on += prob * (m * D_val - f + V[t + 1, 1, new_c])
                else:
                    feasible_on = False
                    break
            if not feasible_on:
                ev_on = NEG_INF

            if ev_on > ev_off:
                V[t, 0, c] = ev_on
                X[t, 0, c] = 1
            else:
                V[t, 0, c] = ev_off
                X[t, 0, c] = 0

            ev_stay = 0.0
            feasible_stay = True
            for D_val, prob in scenarios:
                new_c = c + int(D_val / step)
                if new_c < n_cum:
                    ev_stay += prob * (m * D_val + V[t + 1, 1, new_c])
                else:
                    feasible_stay = False
                    break
            if not feasible_stay:
                ev_stay = NEG_INF

            ev_turn_off = V[t + 1, 0, c]
            if ev_stay >= ev_turn_off:
                V[t, 1, c] = ev_stay
                X[t, 1, c] = 0
            else:
                V[t, 1, c] = ev_turn_off
                X[t, 1, c] = -1

    return V, X


def reshape_week(values, index_name="Hour"):
    arr = np.asarray(values).reshape(7, 24).T
    return pd.DataFrame(arr, index=pd.Index(range(1, 25), name=index_name), columns=DAYS)


def build_terminal_row(V):
    return pd.DataFrame({"state": [0, 1], "V_terminal": [V[-1, 0], V[-1, 1]]})


def schedule_status(states):
    status = []
    for t in range(len(states) - 1):
        if states[t] == 0 and states[t + 1] == 1:
            status.append("ON*")
        elif states[t + 1] == 1:
            status.append("ON")
        else:
            status.append(".")
    return np.array(status)


def hourly_profit(states, margins, demands, startups):
    profit = np.zeros(len(states) - 1)
    for t in range(len(states) - 1):
        if states[t] == 0 and states[t + 1] == 1:
            profit[t] = margins[t] * demands[t] - startups[t]
        elif states[t + 1] == 1:
            profit[t] = margins[t] * demands[t]
    return profit


def build_hourly_schedule_table(states, margins, demands, startups):
    status = schedule_status(states)
    profit = hourly_profit(states, margins, demands, startups)
    rows = []
    for t in range(len(status)):
        rows.append(
            {
                "t": t + 1,
                "day": DAYS[t // 24],
                "hour": t % 24 + 1,
                "status": status[t],
                "margin_eur_per_mwh": margins[t],
                "demand_mw": demands[t],
                "startup_cost_eur": startups[t],
                "hourly_profit_eur": profit[t],
            }
        )
    return pd.DataFrame(rows)


def summarize_case(states, margins, demands, startups, label):
    profit = hourly_profit(states, margins, demands, startups)
    status = schedule_status(states)
    total_mwh = float(sum(demands[t] for t in range(len(status)) if states[t + 1] == 1))
    return {
        "case": label,
        "profit_eur": float(profit.sum()),
        "production_mwh": total_mwh,
        "startups": int((status == "ON*").sum()),
    }


## Chargement des données


In [3]:
margins, demands, startups, stoch_demand = load_data()
margins_24, demands_24, startups_24, ref_V_s0, ref_V_s1, ref_x_s0, ref_x_s1 = load_24h_validation()

overview = pd.DataFrame(
    {
        "series": ["Margins", "Demand", "Startup cost"],
        "min": [margins.min(), demands.min(), startups.min()],
        "max": [margins.max(), demands.max(), startups.max()],
        "mean": [margins.mean(), demands.mean(), startups.mean()],
    }
)
display(overview)


,series,min,max,mean
0,Margins,-5.25,6.73,0.236964
1,Demand,100.00,400.00,211.904762
2,Startup cost,800.00,3600.00,2151.785714


## Validation sur le cas 24 heures

L'énoncé de base exige la reproduction de la valeur optimale `V1(0) ≈ 1652.83`.


In [4]:
V_24, X_24 = solve_base_dp(margins_24, demands_24, startups_24)

validation_summary = pd.DataFrame(
    {
        "metric": ["Calculated V1(0)", "Reference V1(0)", "Absolute gap"],
        "value": [V_24[0, 0], ref_V_s0[0], abs(V_24[0, 0] - ref_V_s0[0])],
    }
)

compare_24h = pd.DataFrame(
    {
        "hour": np.arange(1, 25),
        "margin": margins_24,
        "x_calc_s0": X_24[:, 0],
        "x_ref_s0": ref_x_s0[:24].astype(int),
        "match_s0": X_24[:, 0] == ref_x_s0[:24].astype(int),
        "V_calc_s0": V_24[:24, 0],
        "V_ref_s0": ref_V_s0[:24],
    }
)

display(validation_summary)
display(compare_24h)


,metric,value
0,Calculated V1(0),1652.833977
1,Reference V1(0),1652.833977
2,Absolute gap,0.000000


,hour,margin,x_calc_s0,x_ref_s0,match_s0,V_calc_s0,V_ref_s0
0,1,-1.837175,0,0,True,1652.833977,1652.833977
1,2,4.103765,1,1,True,1652.833977,1652.833977
2,3,1.580391,0,0,True,1284.820767,1284.820767
3,4,0.902039,0,0,True,1284.820767,1284.820767
4,5,-2.906063,0,0,True,1284.820767,1284.820767
5,6,2.766988,1,1,True,1284.820767,1284.820767
6,7,-1.100000,0,0,True,1153.121988,1153.121988
7,8,-0.350000,0,0,True,1153.121988,1153.121988
8,9,2.489049,1,1,True,1153.121988,1153.121988
9,10,2.414172,1,1,True,904.217039,904.217039


## Problème de base sur 168 heures


In [5]:
V_base, X_base = solve_base_dp(margins, demands, startups)
states_base, decisions_base = extract_schedule(X_base, s0=0)

base_summary = pd.DataFrame([summarize_case(states_base, margins, demands, startups, "Base 168h")])
display(base_summary)


,case,profit_eur,production_mwh,startups
0,Base 168h,20699.0,19000.0,8


### Tableaux demandés pour le cas de base

Les matrices ci-dessous donnent les décisions optimales et les fonctions de valeur sur les 168 heures, organisées en grille `24 x 7`.


In [6]:
base_x_s0 = reshape_week(X_base[:, 0])
base_x_s1 = reshape_week(X_base[:, 1])
base_V_s0 = reshape_week(np.round(V_base[:168, 0], 2))
base_V_s1 = reshape_week(np.round(V_base[:168, 1], 2))
base_terminal = build_terminal_row(V_base)

display(Markdown("**Decisions optimales x_t(s=0)**"))
display(base_x_s0)
display(Markdown("**Decisions optimales x_t(s=1)**"))
display(base_x_s1)
display(Markdown("**Fonctions de valeur V_t(s=0)**"))
display(base_V_s0)
display(Markdown("**Fonctions de valeur V_t(s=1)**"))
display(base_V_s1)
display(Markdown("**Condition terminale**"))
display(base_terminal)


**Decisions optimales x_t(s=0)**

,Lun,Mar,Mer,Jeu,Ven,Sam,Dim
Hour,,,,,,,
1,0,0,1,0,0,0,0
2,0,0,0,0,0,1,0
3,0,0,1,0,0,0,0
4,0,0,0,0,0,0,0
5,0,0,0,0,0,1,0
6,0,0,0,1,0,0,0
7,0,0,1,1,0,1,0
8,0,0,0,0,0,1,1
9,0,0,0,0,0,0,1


**Decisions optimales x_t(s=1)**

,Lun,Mar,Mer,Jeu,Ven,Sam,Dim
Hour,,,,,,,
1,0,0,0,0,0,0,-1
2,0,0,0,-1,0,0,-1
3,0,0,0,0,0,0,-1
4,-1,-1,0,0,0,0,-1
5,0,0,0,0,0,0,-1
6,-1,0,0,0,-1,0,0
7,0,0,0,0,-1,0,0
8,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0


**Fonctions de valeur V_t(s=0)**

,Lun,Mar,Mer,Jeu,Ven,Sam,Dim
Hour,,,,,,,
1,20699.0,15995.5,15030.5,11665.5,6195.0,6134.5,2708.5
2,20699.0,15995.5,14839.5,11665.5,6195.0,6134.5,2708.5
3,20699.0,15995.5,14839.5,11665.5,6195.0,6076.5,2708.5
4,20699.0,15995.5,14811.5,11665.5,6195.0,6076.5,2708.5
5,20699.0,15995.5,14811.5,11665.5,6195.0,6076.5,2708.5
6,20699.0,15995.5,14811.5,11665.5,6195.0,5780.5,2708.5
7,20699.0,15995.5,14811.5,10823.5,6195.0,5780.5,2708.5
8,20699.0,15995.5,12404.5,10812.5,6195.0,5230.0,2708.5
9,20699.0,15995.5,12404.5,10812.5,6195.0,4582.0,2560.0


**Fonctions de valeur V_t(s=1)**

,Lun,Mar,Mer,Jeu,Ven,Sam,Dim
Hour,,,,,,,
1,21100.0,16858.0,16030.5,12757.5,7072.5,6934.0,2708.5
2,21108.0,16661.5,15758.5,11665.5,7457.5,7334.5,2708.5
3,20863.0,16474.0,15839.5,11665.5,8182.5,7134.5,2708.5
4,20699.0,15995.5,15495.5,12501.5,8227.5,6878.5,2708.5
5,20705.0,16142.5,15531.5,12543.5,7267.5,7276.5,2708.5
6,20699.0,17357.5,15651.5,13165.5,6195.0,7490.5,3748.0
7,21207.0,17727.5,16311.5,12323.5,6195.0,7580.5,4451.5
8,21069.0,16725.5,14351.5,12163.5,7007.0,7030.0,5108.5
9,21421.5,17955.5,13379.5,12790.5,6278.0,5551.0,4960.0


**Condition terminale**

,state,V_terminal
0,0,0.000000e+00
1,1,-1.000000e+15


In [7]:
base_planning = reshape_week(schedule_status(states_base))
base_hourly_table = build_hourly_schedule_table(states_base, margins, demands, startups)

display(Markdown("**Planning compact ON/OFF**"))
display(base_planning)
display(Markdown("**Table horaire détaillée**"))
display(base_hourly_table)


**Planning compact ON/OFF**

,Lun,Mar,Mer,Jeu,Ven,Sam,Dim
Hour,,,,,,,
1,.,ON,ON,ON,ON,ON,.
2,.,ON,ON,.,ON,ON,.
3,.,ON,ON,.,ON,ON,.
4,.,.,ON,.,ON,ON,.
5,.,.,ON,.,ON,ON,.
6,.,.,ON,ON*,.,ON,.
7,.,.,ON,ON,.,ON,.
8,.,.,ON,ON,.,ON,ON*
9,.,.,ON,ON,.,ON,ON


**Table horaire détaillée**

,t,day,hour,status,margin_eur_per_mwh,demand_mw,startup_cost_eur,hourly_profit_eur
0,1,Lun,1,.,-0.08,100.0,1000.0,0.0
1,2,Lun,2,.,2.45,100.0,1000.0,0.0
2,3,Lun,3,.,1.64,100.0,1000.0,0.0
3,4,Lun,4,.,-5.14,100.0,1000.0,0.0
4,5,Lun,5,.,0.06,100.0,1000.0,0.0
...,...,...,...,...,...,...,...,...
163,164,Dim,20,ON,3.96,200.0,2000.0,792.0
164,165,Dim,21,ON,0.16,200.0,2000.0,32.0
165,166,Dim,22,.,-1.48,150.0,1500.0,0.0
166,167,Dim,23,.,-2.19,150.0,1500.0,0.0


## Extensions


In [8]:
V_ext1, X_ext1 = solve_ext1_min_runtime(margins, demands, startups, min_hours=10)
states_ext1, decisions_ext1, states_detail_ext1 = extract_schedule_ext1(X_ext1, min_hours=10)

V_ext2, X_ext2 = solve_ext2_production_cap(margins, demands, startups, max_mwh=16500)
states_ext2, decisions_ext2, cum_ext2 = extract_schedule_ext2(X_ext2, demands, step=50)

V_ext3, X_ext3 = solve_ext3_stochastic(margins, demands, startups, stoch_demand, max_mwh=16500)
demands_expected = demands.copy()
for h in range(24):
    D1, p1, D2, p2 = stoch_demand[h]
    demands_expected[144 + h] = D1 * p1 + D2 * p2
V_ext3_det, _ = solve_ext2_production_cap(margins, demands_expected, startups, max_mwh=16500)

summary_rows = [
    summarize_case(states_base, margins, demands, startups, "Base"),
    summarize_case(states_ext1, margins, demands, startups, "Ext 1 - min 10h"),
    summarize_case(states_ext2, margins, demands, startups, "Ext 2 - cap 16500"),
    {
        "case": "Ext 3 - stochastic + cap",
        "profit_eur": float(V_ext3[0, 0, 0]),
        "production_mwh": np.nan,
        "startups": np.nan,
    },
    {
        "case": "Ext 3 with E[D]",
        "profit_eur": float(V_ext3_det[0, 0, 0]),
        "production_mwh": np.nan,
        "startups": np.nan,
    },
]
extensions_summary = pd.DataFrame(summary_rows)
extensions_summary["delta_vs_base"] = extensions_summary["profit_eur"] - extensions_summary.loc[0, "profit_eur"]
display(extensions_summary)


,case,profit_eur,production_mwh,startups,delta_vs_base
0,Base,20699.000000,19000.0,8.0,0.000000
1,Ext 1 - min 10h,19785.000000,18800.0,6.0,-914.000000
2,Ext 2 - cap 16500,19764.000000,16400.0,8.0,-935.000000
3,Ext 3 - stochastic + cap,19459.218628,NaN,NaN,-1239.781372
4,Ext 3 with E[D],19764.000000,NaN,NaN,-935.000000


In [9]:
ext1_planning = reshape_week(schedule_status(states_ext1))
ext2_planning = reshape_week(schedule_status(states_ext2))
ext2_cum_table = pd.DataFrame({"t": np.arange(1, 169), "cum_step_50": cum_ext2[1:], "cum_mwh": cum_ext2[1:] * 50})

display(Markdown("**Planning extension 1**"))
display(ext1_planning)
display(Markdown("**Planning extension 2**"))
display(ext2_planning)
display(Markdown("**Cumul de production extension 2**"))
display(ext2_cum_table)

display(Markdown("**Note extension 3**"))
display(Markdown(
    "Avec demande stochastique le dimanche, le résultat est une politique conditionnelle et non un planning fixe unique. "
    f"Le profit espéré optimal vaut **{V_ext3[0, 0, 0]:.2f} EUR**, contre **{V_ext3_det[0, 0, 0]:.2f} EUR** si l'on remplace la demande aléatoire par son espérance."
))


**Planning extension 1**

,Lun,Mar,Mer,Jeu,Ven,Sam,Dim
Hour,,,,,,,
1,.,ON,ON,ON,ON,ON,.
2,.,ON,ON,ON,ON,ON,.
3,.,ON,ON,ON,ON,ON,.
4,.,.,ON,ON,ON,ON,.
5,.,.,ON,ON,ON,ON,.
6,.,.,ON,ON,.,ON,.
7,.,.,ON,ON,.,ON,.
8,.,.,ON,ON,.,ON,ON*
9,.,.,ON,ON,.,ON,ON


**Planning extension 2**

,Lun,Mar,Mer,Jeu,Ven,Sam,Dim
Hour,,,,,,,
1,.,ON,ON,ON,.,.,.
2,.,ON,ON,.,.,.,.
3,.,ON,ON,.,.,.,.
4,.,.,ON,.,.,.,.
5,.,.,ON,.,.,ON*,.
6,.,.,ON,ON*,.,ON,.
7,.,.,ON,ON,.,ON,.
8,.,.,ON,ON,.,ON,ON*
9,.,.,ON,ON,.,ON,ON


**Cumul de production extension 2**

,t,cum_step_50,cum_mwh
0,1,0,0
1,2,0,0
2,3,0,0
3,4,0,0
4,5,0,0
...,...,...,...
163,164,328,16400
164,165,328,16400
165,166,328,16400
166,167,328,16400


**Note extension 3**

Avec demande stochastique le dimanche, le résultat est une politique conditionnelle et non un planning fixe unique. Le profit espéré optimal vaut **19459.22 EUR**, contre **19764.00 EUR** si l'on remplace la demande aléatoire par son espérance.

## Export des résultats

Le classeur ci-dessous facilite la remise des tableaux de résultats.


In [10]:
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    validation_summary.to_excel(writer, sheet_name="Validation24h", index=False)
    compare_24h.to_excel(writer, sheet_name="Validation24h_Compare", index=False)
    base_summary.to_excel(writer, sheet_name="BaseSummary", index=False)
    base_x_s0.to_excel(writer, sheet_name="Base_X_s0")
    base_x_s1.to_excel(writer, sheet_name="Base_X_s1")
    base_V_s0.to_excel(writer, sheet_name="Base_V_s0")
    base_V_s1.to_excel(writer, sheet_name="Base_V_s1")
    base_terminal.to_excel(writer, sheet_name="Base_Terminal", index=False)
    base_planning.to_excel(writer, sheet_name="Base_Planning")
    base_hourly_table.to_excel(writer, sheet_name="Base_Hourly", index=False)
    ext1_planning.to_excel(writer, sheet_name="Ext1_Planning")
    ext2_planning.to_excel(writer, sheet_name="Ext2_Planning")
    ext2_cum_table.to_excel(writer, sheet_name="Ext2_Cum", index=False)
    extensions_summary.to_excel(writer, sheet_name="Extensions_Summary", index=False)

print(f"Results exported to: {OUTPUT_XLSX}")


Results exported to: D:\realoption\dispatch_results.xlsx
